Join df_obs ↔ cmip_station_daily

In [ ]:
import polars as pl
from datetime import date
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

DF_OBS_PATH   = "../data/weatherstation_data/df_obs.parquet"
CMIP_ST_PATH  = "../data/weatherstation_data/cmip_station_daily_2017_2020-11-13.parquet"

df_obs  = pl.read_parquet(DF_OBS_PATH)
cmip_st = pl.read_parquet(CMIP_ST_PATH)

# inner join по (STATION, DATE)
df_eval = df_obs.join(cmip_st, on=["STATION", "DATE"], how="inner")

print("df_obs rows:", df_obs.height)
print("cmip_st rows:", cmip_st.height)
print("df_eval rows:", df_eval.height)
print("coverage:", df_eval.height / df_obs.height)


df_obs rows: 2454571
cmip_st rows: 2917192
df_eval rows: 2452735
coverage: 0.9992520077846597


Разделение на val / test

In [ ]:
df_val = df_eval.filter(
    pl.col("DATE").is_between(date(2017,1,1), date(2018,12,31))
)

df_test = df_eval.filter(
    pl.col("DATE").is_between(date(2019,1,1), date(2020,11,13))
)

print(
    "val:", df_val.height, 
    "events:", df_val.select(pl.col("y_true").sum()).item(),
    "rate:", df_val.select(pl.col("y_true").mean()).item()
)
print(
    "test:", df_test.height, 
    "events:", df_test.select(pl.col("y_true").sum()).item(),
    "rate:", df_test.select(pl.col("y_true").mean()).item()
)


val: 1260954 events: 71345 rate: 0.05658017659644999
test: 1191781 events: 71088 rate: 0.05964854281113728


### Baseline “CMIP threshold”

- на val подбираем порог T для cmip_wind,

- фиксируем его,

- считаем метрики только на test.

In [ ]:
x_val = df_val["cmip_wind"].to_numpy()
y_val = df_val["y_true"].to_numpy()

x_test = df_test["cmip_wind"].to_numpy()
y_test = df_test["y_true"].to_numpy()


Подбор порога

Берём квантили — этого достаточно для baseline.

In [ ]:
candidates = np.quantile(x_val, np.linspace(0.80, 0.999, 200))

best_T = None
best_f1 = -1.0

for T in candidates:
    y_pred = (x_val > T).astype(int)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_T = T

best_T, best_f1
# 2 mins


(9.50539086341858, 0.1947260080192775)

Метрики baseline на test (главное для статьи)

Это основной baseline, с которым будем сравнивать CNN.

In [ ]:
y_pred_test = (x_test > best_T).astype(int)

precision = precision_score(y_test, y_pred_test, zero_division=0)
recall    = recall_score(y_test, y_pred_test, zero_division=0)
f1        = f1_score(y_test, y_pred_test, zero_division=0)

precision, recall, f1


(0.14645999952815722, 0.26198514517218097, 0.18788493258477973)

# LogReg

In [ ]:
DF_OBS_PATH   = "data/weatherstation_data/df_obs_2000_2020_Russia.parquet"
CMIP_ST_PATH  = "data/weatherstation_data/cmip_station_2000_2020_Russia.parquet"

df_obs  = pl.read_parquet(DF_OBS_PATH)
cmip_st = pl.read_parquet(CMIP_ST_PATH)

# inner join по (STATION, DATE)
df_eval = df_obs.join(cmip_st, on=["STATION", "DATE"], how="inner")

In [ ]:
import polars as pl
import numpy as np
from datetime import date
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

# --- split ---
df_train = df_eval.filter(pl.col("DATE").is_between(date(2000,1,1), date(2016,12,31)))
df_val   = df_eval.filter(pl.col("DATE").is_between(date(2017,1,1), date(2018,12,31)))
df_test  = df_eval.filter(pl.col("DATE").is_between(date(2019,1,1), date(2020,11,13)))

# --- feature engineering: month -> sin/cos ---
def add_season_feats(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("DATE").dt.month().alias("month"),
    ]).with_columns([
        (2*np.pi*pl.col("month")/12).sin().alias("m_sin"),
        (2*np.pi*pl.col("month")/12).cos().alias("m_cos"),
    ])

train = add_season_feats(df_train).drop_nulls(["cmip_wind", "y_true"])
val   = add_season_feats(df_val).drop_nulls(["cmip_wind", "y_true"])
test  = add_season_feats(df_test).drop_nulls(["cmip_wind", "y_true"])

# --- build matrices ---
feat_cols = ["cmip_wind", "m_sin", "m_cos"]

X_train = train.select(feat_cols).to_numpy()
y_train = train.select("y_true").to_numpy().ravel()

X_val = val.select(feat_cols).to_numpy()
y_val = val.select("y_true").to_numpy().ravel()

X_test = test.select(feat_cols).to_numpy()
y_test = test.select("y_true").to_numpy().ravel()

# --- fit logistic regression on TRAIN ---
clf = LogisticRegression(
    solver="lbfgs",
    max_iter=200,
    class_weight="balanced",   # важно при дисбалансе
    n_jobs=None
)
clf.fit(X_train, y_train)

# --- choose probability threshold on VAL (optimize F1) ---
p_val = clf.predict_proba(X_val)[:, 1]

thr_candidates = np.quantile(p_val, np.linspace(0.50, 0.999, 200))
best_thr, best_f1 = None, -1.0

for thr in thr_candidates:
    y_pred = (p_val >= thr).astype(int)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

print("best_thr:", float(best_thr), "best_val_f1:", float(best_f1))

# --- evaluate on TEST ---
p_test = clf.predict_proba(X_test)[:, 1]
y_pred_test = (p_test >= best_thr).astype(int)

prec = precision_score(y_test, y_pred_test, zero_division=0)
rec  = recall_score(y_test, y_pred_test, zero_division=0)
f1   = f1_score(y_test, y_pred_test, zero_division=0)

print("LOGREG baseline on TEST:", "precision", prec, "recall", rec, "f1", f1)

# (опционально) sanity: коэффициенты интерпретируемости
print("coef:", dict(zip(feat_cols, clf.coef_[0])))
print("intercept:", float(clf.intercept_[0]))


best_thr: 0.6442717735296724 best_val_f1: 0.19280042959610294
LOGREG baseline on TEST: precision 0.1452593534159303 recall 0.26685235201440466 f1 0.1881179282236392
coef: {'cmip_wind': 0.14921937559302434, 'm_sin': 0.20330399382216313, 'm_cos': 0.21988467741058804}
intercept: -0.894807518787345
